In [ ]:
### naprawa + deduplikacja

import re
import pandas as pd
from io import StringIO

input_file = 'hathi_articles.csv'
final_file = 'hathi_articles_fixed.csv'

print("1/4 Naprawa połamanych wierszy...")

with open(input_file, 'r', encoding='utf-8') as f:
    content = f.read()

# fix – wszystkie końce linii (Windows, Unix, Mac) -> spacja, chyba że nowa linia zaczyna się od "keyword,"
fixed = re.sub(r'(?:\r\n|\r|\n)(?!keyword,)', ' ', content)

# dodatkowo: zamienia wszystkie cudzysłowy wewnątrz pól na pojedyncze ' (żeby nie psuły parsowania)
fixed = fixed.replace('"', "'")

print("2/4 Wczytanie do pandas (z pamięci, engine='python' – odporny na błędy)...")

# wczytanie z pamięci, używając silnika python – ignoruje źle zamknięte cudzysłowy
df = pd.read_csv(
    StringIO(fixed),
    engine='python',           ## kluczowe – nie wywala się na błędach
    on_bad_lines='skip',       ## pomija całkowicie zepsute linie (będzie ich < 10)
    dtype=str                  ## wszystko jako string
)

print(f"   Wczytano {len(df):,} rekordów (po skipie złych linii)")

# przywrócenie poprawnych nazw kolumn (na wszelki wypadek)
df.columns = ['keyword', 'title', 'authors', 'publisher', 'published_date', 'description', 'hathi_id']

print("3/4 Deduplikacja (title + authors)...")
df.drop_duplicates(subset=['title', 'authors'], keep='first', inplace=True)
print(f"   Po deduplikacji: {len(df):,} rekordów")

print("4/4 Zapis finalnego czystego pliku...")
df.to_csv(final_file, index=False, encoding='utf-8')
print(f"\n--> {final_file} jest czysty, deduplikowany i gotowy.")
print(f"   Liczba finalnych rekordów: {len(df):,}")